# AuroraCart at a Crossroads — Exploratory Data Analysis

**Case:** AuroraCart is a fictional Indian omnichannel lifestyle retailer. Leadership
has watched revenue climb for three straight years and is preparing an investment
pitch to the board built on that growth story. The board's actual question is
narrower: *should the pattern of that growth be funded for another twelve months?*

This notebook is the analytical groundwork for the dashboard
(`src/auroracart/dashboard.py`) and the executive story
(`deliverables/AuroraCart_Executive_Story.pptx`) that answer it.

**What this notebook does, in order:**
1. Audits the raw export for data-quality problems and documents every fix.
2. Builds the KPI vocabulary the rest of the analysis shares.
3. Tests whether revenue growth is creating value — including against the two
   dated interventions the case names: **Accelerate 2.0** (July 2024) and the
   **logistics restructure** (January 2025).
4. Decomposes the margin decline into product mix versus within-category erosion,
   because those two diagnoses imply opposite responses.
5. Drills into the levers: category mix and cost structure, promotions and discount
   tolerance, customer segments, acquisition channels, delivery operations.
6. Ranks every candidate driver by how much it actually separates profitability.
7. Closes with the synthesis, the recommendations, and the limitations.

> Cleaning and feature engineering live in `auroracart.data_prep`, the
> case-question metrics in `auroracart.analysis`, and chart styling in
> `auroracart.viz_theme`. All three are imported here **and** by the dashboard and
> the deck builder, so no artifact can drift from another.

**Two conventions used throughout**, both argued in `docs/methodology.md`:
margin is always `sum(profit) / sum(revenue)` and never the mean of order-level
margins; and cancelled orders leave the revenue denominator but stay in the
operational one.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from auroracart import analysis as A
from auroracart.data_prep import (
    load_data, kpi_summary, CATEGORY_ORDER, REGION_ORDER, FULFILLMENT_ORDER,
    PROMOTION_ORDER, SEGMENT_ORDER,
)
from auroracart.paths import RAW_DATA_PATH
from auroracart.viz_theme import CATEGORICAL, SEQUENTIAL_BLUE, STATUS, INK, finalize

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


## 1. Data-quality audit (before any cleaning)

Load the file exactly as exported so every problem is visible before we touch it.


In [2]:
raw = pd.read_csv(RAW_DATA_PATH)
print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(3)


Raw shape: 15,000 rows x 50 columns


,Order_ID,Customer_ID,Order_Date,Year,Quarter,Month,Month_Name,Weekday,Weekend_Flag,Region,State,City,Urban_Tier,Customer_Segment,Age_Group,Gender,Membership_Type,New_vs_Returning,Customer_Tenure_Months,Acquisition_Channel,Loyalty_Status,Category,Subcategory,Brand_Tier,Price_Band,Fulfillment_Mode,Quantity,Unit_Price,Gross_Revenue,Discount_Percentage,Discount_Amount,Net_Revenue,Product_Cost,Delivery_Cost,Marketing_Cost,Operating_Cost,Total_Cost,Profit,Profit_Margin,Promotion_Type,Coupon_Used,Expected_Delivery_Days,Delivery_Time_Days,Delivery_Delay_Days,On_Time_Flag,Return_Flag,Cancellation_Flag,Customer_Rating,Complaint_Flag,Days_Since_Last_Purchase
0,O0005692,C002150,2024-07-04,2024,Q3,7,July,Thursday,No,Central,Maharashtra,Nagpur,Tier 2,Family,35-44,Female,Silver,New,0,Marketplace Ads,New,Beauty & Personal Care,Skincare,Value,Mass,Express Delivery,1,"1,211.82","1,211.82",8.92,108.09,"1,103.72",497.33,272.72,120.92,123.56,"1,014.54",89.18,8.08,NaN,Yes,1.80,2.35,0.55,No,No,No,5.00,No,NaN
1,O0005916,C002276,2024-07-20,2024,Q3,7,July,Saturday,Yes,East,West Bengal,Kolkata,Tier 1,Family,35-44,Female,NaN,New,0,Paid Social,New,Fashion,Men,Mid-Market,Mass,Standard Delivery,2,"2,960.09","5,920.18",17.90,"1,059.71",0.00,0.00,0.00,436.28,45.00,481.28,-481.28,NaN,Festival Sale,Yes,3.00,1.72,0.00,Yes,No,Yes,NaN,No,NaN
2,O0010767,C000543,2025-05-22,2025,Q2,5,May,Thursday,No,South,Telangana,Hyderabad,Tier 1,Value Seeker,25-34,Female,Silver,Returning,23,Paid Social,Developing,Electronics,Wearables,Mid-Market,Upper-Mid,Standard Delivery,2,"12,341.66","24,683.32",13.00,"3,208.83","21,474.49","18,735.62",451.44,"1,204.72","1,019.88","21,411.66",62.82,0.29,NaN,Yes,3.00,2.85,0.00,Yes,No,No,4.10,No,89.00


In [3]:
# 1a. Exact duplicate rows — same order captured twice in the export
n_dupes = raw.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes}")
raw[raw.duplicated(keep=False)].sort_values("Order_ID").head(4)


Fully duplicated rows: 30


,Order_ID,Customer_ID,Order_Date,Year,Quarter,Month,Month_Name,Weekday,Weekend_Flag,Region,State,City,Urban_Tier,Customer_Segment,Age_Group,Gender,Membership_Type,New_vs_Returning,Customer_Tenure_Months,Acquisition_Channel,Loyalty_Status,Category,Subcategory,Brand_Tier,Price_Band,Fulfillment_Mode,Quantity,Unit_Price,Gross_Revenue,Discount_Percentage,Discount_Amount,Net_Revenue,Product_Cost,Delivery_Cost,Marketing_Cost,Operating_Cost,Total_Cost,Profit,Profit_Margin,Promotion_Type,Coupon_Used,Expected_Delivery_Days,Delivery_Time_Days,Delivery_Delay_Days,On_Time_Flag,Return_Flag,Cancellation_Flag,Customer_Rating,Complaint_Flag,Days_Since_Last_Purchase
4780,O0001208,C000206,2023-05-19,2023,Q2,5,May,Friday,No,West,Maharashtra,Mumbai,Tier 1,Family,45-54,Female,Gold,Returning,2,Organic Search,Loyal,Sports & Fitness,Nutrition Accessories,Value,Budget,Express Delivery,2,448.63,897.27,15.47,138.81,758.46,431.76,269.42,6.83,122.02,830.04,-71.58,-9.44,Category Offer,Yes,1.00,2.97,1.97,No,No,No,4.00,No,36.00
11289,O0001208,C000206,2023-05-19,2023,Q2,5,May,Friday,No,West,Maharashtra,Mumbai,Tier 1,Family,45-54,Female,Gold,Returning,2,Organic Search,Loyal,Sports & Fitness,Nutrition Accessories,Value,Budget,Express Delivery,2,448.63,897.27,15.47,138.81,758.46,431.76,269.42,6.83,122.02,830.04,-71.58,-9.44,Category Offer,Yes,1.00,2.97,1.97,No,No,No,4.00,No,36.00
3835,O0001752,C000576,2023-07-19,2023,Q3,7,July,Wednesday,No,West,Gujarat,Ahmedabad,Tier 1,Occasional,35-44,Male,Gold,Returning,1,Direct,Loyal,Beauty & Personal Care,Grooming,Mid-Market,Mass,Standard Delivery,1,"1,828.09","1,828.09",3.21,58.68,"1,769.41",916.24,194.19,7.96,184.70,"1,303.10",466.31,26.35,Category Offer,No,3.00,4.59,1.59,No,No,No,3.10,Yes,27.00
4296,O0001752,C000576,2023-07-19,2023,Q3,7,July,Wednesday,No,West,Gujarat,Ahmedabad,Tier 1,Occasional,35-44,Male,Gold,Returning,1,Direct,Loyal,Beauty & Personal Care,Grooming,Mid-Market,Mass,Standard Delivery,1,"1,828.09","1,828.09",3.21,58.68,"1,769.41",916.24,194.19,7.96,184.70,"1,303.10",466.31,26.35,Category Offer,No,3.00,4.59,1.59,No,No,No,3.10,Yes,27.00


In [4]:
# 1b. Free-text categories typed inconsistently — these are the SAME category, not new ones
print("Category, raw unique values:")
print(raw["Category"].unique())
print()
print("Acquisition_Channel, raw unique values:")
print(raw["Acquisition_Channel"].unique())


Category, raw unique values:
<StringArray>
[  'Beauty & Personal Care',                  'Fashion',
              'Electronics',         'Sports & Fitness',
           'Home & Kitchen',           'Home & kitchen',
 'Beauty and Personal Care']
Length: 7, dtype: str

Acquisition_Channel, raw unique values:
<StringArray>
['Marketplace Ads',     'Paid Social',           'Email',        'Referral',
  'Organic Search',  'organic search',          'Direct',     'Paid social']
Length: 8, dtype: str


In [5]:
# 1c. Missing values — for each, we check WHY before deciding how to handle it
missing = raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing.to_frame("missing_count").assign(pct=lambda d: (d["missing_count"] / len(raw) * 100).round(1))


,missing_count,pct
Days_Since_Last_Purchase,6790,45.30
Promotion_Type,5636,37.60
Membership_Type,4757,31.70
Customer_Rating,603,4.00
Profit_Margin,428,2.90
Age_Group,75,0.50


**What each missing field actually means** (verified against related columns, not assumed):

| Column | Missing | Root cause | Decision |
|---|---|---|---|
| `Days_Since_Last_Purchase` | 6,790 | 100% of these rows are `New_vs_Returning == 'New'` — a first-time buyer has no prior purchase to count days since. This is *structural*, not broken data. | Leave as `NaN`; never impute a fake recency for a first-time customer. |
| `Promotion_Type` | 5,636 | The order simply wasn't placed under any promotion. | Recode to explicit label `"No Promotion"` so it participates in group-bys instead of silently vanishing. |
| `Membership_Type` | 4,757 | The customer holds no paid membership tier. | Recode to `"No Membership"`. |
| `Customer_Rating` | 603 | 428 are cancelled orders (never delivered, nothing to rate); the remaining 175 are orders where the customer simply didn't leave a rating. | Leave as `NaN`; averages use `skipna` so this never silently drags ratings down. |
| `Profit_Margin` | 428 | Exactly the 428 cancelled orders, where `Net_Revenue == 0` makes margin undefined (division by zero), confirmed 1:1 below. | Leave as `NaN`; recomputed safely as `Net_Margin_Pct` in `data_prep.py` (`NaN` when revenue is 0). |
| `Age_Group` | 75 | No discernible pattern — looks like genuine data entry gaps (0.5% of rows). | Recode to `"Unknown"` rather than drop the row (the rest of the order is still usable). |

We verify the two most consequential claims above (recency ↔ New customers, and
margin ↔ cancellations) directly:


In [6]:
print("Days_Since_Last_Purchase missing, by New_vs_Returning:")
print(raw.loc[raw["Days_Since_Last_Purchase"].isna(), "New_vs_Returning"].value_counts())
print()
print("Rows where Net_Revenue == 0:", (raw["Net_Revenue"] == 0).sum())
print("...of which cancelled:", ((raw["Net_Revenue"] == 0) & (raw["Cancellation_Flag"] == "Yes")).sum())


Days_Since_Last_Purchase missing, by New_vs_Returning:
New_vs_Returning
New    6790
Name: count, dtype: int64

Rows where Net_Revenue == 0: 428
...of which cancelled: 428


## 2. Apply the cleaning pipeline

Everything above is fixed by `data_prep.load_data()`: duplicates dropped, category /
channel spelling standardized, structural nulls recoded, calendar and KPI columns
engineered. The dashboard calls the exact same function.


In [7]:
df = load_data()
print(f"Clean shape: {df.shape[0]:,} rows x {df.shape[1]} columns  "
      f"({raw.shape[0] - df.shape[0]} duplicate rows removed)")
df[["Order_Date", "Category", "Acquisition_Channel", "Promotion_Type", "Membership_Type", "Net_Margin_Pct"]].head(5)


Clean shape: 14,970 rows x 54 columns  (30 duplicate rows removed)


,Order_Date,Category,Acquisition_Channel,Promotion_Type,Membership_Type,Net_Margin_Pct
0,2024-07-04,Beauty & Personal Care,Marketplace Ads,No Promotion,Silver,8.08
1,2024-07-20,Fashion,Paid Social,Festival Sale,No Membership,NaN
2,2025-05-22,Electronics,Paid Social,No Promotion,Silver,0.29
3,2024-10-02,Sports & Fitness,Email,Festival Sale,Silver,25.28
4,2024-03-18,Beauty & Personal Care,Referral,Member Offer,Gold,32.51


In [8]:
kpis = kpi_summary(df)
summary_rows = [
    ("Net Revenue", f"₹{kpis['net_revenue']:,.0f}"),
    ("Profit", f"₹{kpis['profit']:,.0f}"),
    ("Overall Margin", f"{kpis['margin_pct']:.1f}%"),
    ("Orders (valid + cancelled)", f"{kpis['orders']:,}"),
    ("Average Order Value", f"₹{kpis['aov']:,.0f}"),
    ("Unique Customers", f"{kpis['customers']:,}"),
    ("Avg Customer Rating", f"{kpis['avg_rating']:.2f} / 5"),
    ("On-Time Delivery Rate", f"{kpis['on_time_rate']:.1f}%"),
    ("Return Rate", f"{kpis['return_rate']:.1f}%"),
    ("Cancellation Rate", f"{kpis['cancellation_rate']:.1f}%"),
    ("Complaint Rate", f"{kpis['complaint_rate']:.1f}%"),
]
pd.DataFrame(summary_rows, columns=["KPI", "Value (Jan 2023 – Dec 2025, full period)"])


,KPI,"Value (Jan 2023 – Dec 2025, full period)"
0,Net Revenue,"₹130,758,067"
1,Profit,"₹11,604,453"
2,Overall Margin,8.9%
3,Orders (valid + cancelled),"14,970"
4,Average Order Value,"₹8,991"
5,Unique Customers,"6,778"
6,Avg Customer Rating,4.20 / 5
7,On-Time Delivery Rate,43.1%
8,Return Rate,7.2%
9,Cancellation Rate,2.9%


Three years in, at a glance: **₹130.8M** of net revenue at an **8.9% margin**, a
**43% on-time delivery rate** and an **8.7% complaint rate**. None of that is
visible in a "revenue is up" slide.

Two of those three headline numbers turn out to be misleading in the specific way
the case warns about — they are averages over populations that do not behave
alike. The rest of this notebook takes them apart.

## 3. Context → Tension: is growth creating value?

**Context:** AuroraCart's revenue nearly doubled between 2023 and 2025.
**Tension:** did profit — and profit *margin* — grow with it?


In [9]:
valid = df[df["Is_Valid_Revenue"]]
yearly = (
    valid.groupby("Year")
    .agg(Net_Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "count"))
    .assign(Margin_Pct=lambda d: d["Profit"] / d["Net_Revenue"] * 100)
    .reset_index()
)
yearly


,Year,Net_Revenue,Profit,Orders,Margin_Pct
0,2023,"29,284,488.51","3,618,542.15",3542,12.36
1,2024,"43,478,062.16","3,874,707.15",4832,8.91
2,2025,"57,995,515.88","4,111,203.73",6169,7.09


In [10]:
fig = px.bar(yearly, x="Year", y="Net_Revenue", text_auto=".2s",
             title="Revenue nearly doubled 2023 → 2025", color_discrete_sequence=[CATEGORICAL[0]])
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="Net Revenue (₹)", xaxis_title=None, xaxis=dict(type="category"))
finalize(fig, height=380).show()


In [11]:
fig = px.line(yearly, x="Year", y="Margin_Pct", markers=True,
              title="...while profit margin fell by more than a third",
              color_discrete_sequence=[STATUS["critical"]])
fig.update_traces(line=dict(width=3), marker=dict(size=10))
fig.update_layout(yaxis_title="Profit Margin (%)", xaxis_title=None, xaxis=dict(type="category"),
                   yaxis=dict(rangemode="tozero"))
finalize(fig, height=380).show()


**Evidence:** overall margin went **12.4% → 8.9% → 7.1%** while revenue grew ~98%.
Profit itself is still rising (₹3.6M → ₹4.1M), but far slower than revenue — every
incremental rupee of revenue is worth less than the rupee before it. This is the
central tension: **AuroraCart is buying growth with margin.** The next sections find
out which levers are doing the buying.


## 4. Did *Accelerate 2.0* create value?

The case tells us management launched a growth programme in **July 2024**: more
acquisition spend, deeper promotion, further Tier 2 expansion. That is a dated,
pre-declared intervention, which makes it a fair thing to measure a before/after
against — unlike a break we might find by hunting the series for a wiggle.

The comparison is **per month**, because the two windows need not be equal length
and a longer window always wins on totals.

In [12]:
accelerate = A.accelerate_comparison(df)
accelerate[[
    "Orders_Per_Month", "Revenue_Per_Month", "Profit_Per_Month", "Margin_Pct",
    "Avg_Discount_Pct", "AOV", "New_Customer_Share_Pct",
]].round(2).T

Era,Before Accelerate 2.0,After Accelerate 2.0
Orders_Per_Month,305.06,502.89
Revenue_Per_Month,"2,622,913.24","4,641,423.79"
Profit_Per_Month,"300,719.56","343,972.28"
Margin_Pct,11.47,7.41
Avg_Discount_Pct,11.00,14.93
AOV,"8,598.15","9,229.52"
New_Customer_Share_Pct,37.63,49.90


In [13]:
growth = pd.DataFrame({
    "Measure": ["Net revenue per month", "Contribution per month"],
    "Before": [accelerate["Revenue_Per_Month"].iloc[0], accelerate["Profit_Per_Month"].iloc[0]],
    "After": [accelerate["Revenue_Per_Month"].iloc[1], accelerate["Profit_Per_Month"].iloc[1]],
})
growth["Change_Pct"] = (growth["After"] / growth["Before"] - 1) * 100
print(growth.assign(Before=lambda d: d["Before"] / 1e6, After=lambda d: d["After"] / 1e6)
      .rename(columns={"Before": "Before (₹M)", "After": "After (₹M)"}).round(2).to_string(index=False))

               Measure  Before (₹M)  After (₹M)  Change_Pct
 Net revenue per month         2.62        4.64       76.96
Contribution per month         0.30        0.34       14.38


**Evidence:** revenue per month rose **+77%**; contribution per month rose **+14%**.
Margin fell 11.5% → 7.4%, average discount rose 11.0% → 14.9%, and new customers
went from 37.6% to 49.9% of orders.

The programme did what it said it would — more customers, more volume, more Tier 2.
It also discounted four points deeper to do it, and the incremental revenue arrived
at roughly a third of the margin the base business was earning. **Growth was
purchased.** The question is what it was purchased *with*.

## 5. Is this a mix problem or an erosion problem?

Before drilling anywhere, one fork decides which drill to pick up:

* **Mix shift** — we now sell a different basket, weighted toward lower-margin
  categories. The response is merchandising.
* **Within-category erosion** — the same basket earns less than it did. The
  response is pricing and cost.

They are separable. Re-weight the base year's category margins with the final
year's revenue mix to isolate the mix effect; hold the mix fixed and let the
margins move to isolate the rate effect.

In [14]:
decomposition = A.decompose_margin_change(df)
print(f"{decomposition.base_year} margin:            {decomposition.base_margin:6.2f}%")
print(f"  mix shift:              {decomposition.mix_effect:+6.2f} pts")
print(f"  within-category rate:   {decomposition.rate_effect:+6.2f} pts")
print(f"  interaction (residual): {decomposition.interaction:+6.2f} pts")
print(f"{decomposition.final_year} margin:            {decomposition.final_margin:6.2f}%")
print(f"\nRate erosion accounts for {decomposition.rate_share_pct:.0f}% of the movement.")

2023 margin:             12.36%
  mix shift:               -0.41 pts
  within-category rate:    -4.75 pts
  interaction (residual):  -0.12 pts
2025 margin:              7.09%

Rate erosion accounts for 92% of the movement.


In [15]:
# The same split, seen directly: revenue mix barely moved, category margins did.
yearly = df[df["Is_Valid_Revenue"]].copy()
yearly["Year"] = yearly["Order_Date"].dt.year
grouped = yearly.groupby(["Year", "Category"], observed=True).agg(
    Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"))
mix = grouped["Revenue"].unstack()
print("Revenue mix by year (%):")
print((mix.div(mix.sum(axis=1), axis=0) * 100).round(1).to_string())
print("\nMargin within category by year (%):")
print(((grouped["Profit"] / grouped["Revenue"] * 100).unstack()).round(1).to_string())

Revenue mix by year (%):
Category  Electronics  Home & Kitchen  Fashion  Sports & Fitness  Beauty & Personal Care
Year                                                                                    
2023            49.40           22.00    12.20              9.50                    6.90
2024            50.80           21.60    10.70             10.10                    6.90
2025            51.10           20.10    11.20             10.20                    7.50

Margin within category by year (%):
Category  Electronics  Home & Kitchen  Fashion  Sports & Fitness  Beauty & Personal Care
Year                                                                                    
2023            -0.30           22.30    29.10             21.80                   28.90
2024            -4.40           20.20    27.30             19.00                   28.30
2025            -7.50           21.40    26.10             16.80                   26.70


**Evidence:** of the 5.27-point fall, **mix explains −0.41 points and
within-category erosion explains −4.75** — 92% of the movement. Holding the 2023
mix fixed, 2025 margin would still land at 7.6%.

The tables underneath show why: Electronics' share of revenue moved only 49.4% →
51.1%, while its margin went **−0.3% → −7.5%**. Every category lost ground, but
one lost it at scale and from an already-negative base.

That settles the fork. This is a pricing and cost problem, and the next sections
go looking for it inside Electronics.

## 6. Investigation: category mix and cost structure

Electronics dominates revenue. Does it dominate profit too?


In [16]:
cat = (
    valid.groupby("Category", observed=True)
    .agg(Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "count"))
    .assign(Margin_Pct=lambda d: d["Profit"] / d["Revenue"] * 100)
    .reindex(CATEGORY_ORDER)
)
cat


,Revenue,Profit,Orders,Margin_Pct
Category,,,,
Electronics,"66,153,291.99","-3,233,053.21",3036,-4.89
Home & Kitchen,"27,459,829.22","5,815,315.49",3757,21.18
Fashion,"14,715,992.39","4,002,605.34",2980,27.20
Sports & Fitness,"13,087,189.84","2,434,448.03",2095,18.60
Beauty & Personal Care,"9,341,763.11","2,585,137.38",2675,27.67


In [17]:
fig = px.bar(cat.reset_index(), x="Revenue", y="Category", orientation="h",
             title="Electronics: ~half of all revenue", color_discrete_sequence=[CATEGORICAL[0]])
fig.update_layout(yaxis=dict(categoryorder="total ascending"), xaxis_title="Net Revenue (₹)", yaxis_title=None)
finalize(fig, height=340).show()

fig2 = px.bar(cat.reset_index(), x="Margin_Pct", y="Category", orientation="h",
              title="...but the only category running a NEGATIVE margin",
              color="Margin_Pct", color_continuous_scale=[STATUS["critical"], INK["grid"], STATUS["good"]],
              color_continuous_midpoint=0)
fig2.update_layout(yaxis=dict(categoryorder="array", categoryarray=cat.sort_values("Margin_Pct").index[::-1]),
                    xaxis_title="Profit Margin (%)", yaxis_title=None, coloraxis_showscale=False)
fig2.add_vline(x=0, line_color=INK["axis"])
finalize(fig2, height=340).show()


**Evidence:** Electronics generates **₹66.2M — 50.6% of net revenue — at a −4.9%
margin**, losing ₹3.23M of contribution. Home & Kitchen, Fashion, Sports & Fitness
and Beauty & Personal Care all run comfortably profitable (18.6–27.7%) on far less
revenue. Electronics is not a small drag: at half the revenue base it is pulling
the company-wide margin down on its own.

**Why might Electronics be underwater?** Three candidates, each testable here:
(a) disproportionately heavy discounting to stay price-competitive, (b) high unit
values inflating delivery cost per order, or (c) merchandise cost set too close to
the price at the point of sourcing. We check all three next.

In [18]:
disc = valid.groupby("Category", observed=True)["Discount_Percentage"].mean().reindex(CATEGORY_ORDER)
cost_share = (valid.groupby("Category", observed=True)
              .apply(lambda d: (d["Delivery_Cost"] + d["Marketing_Cost"] + d["Operating_Cost"]).sum() / d["Net_Revenue"].sum() * 100,
                     include_groups=False)
              .reindex(CATEGORY_ORDER))
pd.DataFrame({"Avg_Discount_Pct": disc, "NonProduct_Cost_Pct_of_Revenue": cost_share})


,Avg_Discount_Pct,NonProduct_Cost_Pct_of_Revenue
Category,,
Electronics,13.37,10.79
Home & Kitchen,13.54,13.69
Fashion,13.46,15.60
Sports & Fitness,13.27,14.40
Beauty & Personal Care,13.51,17.91


In [19]:
# Where each rupee of net revenue goes, by category — the *why* behind the margin bar.
costs = A.cost_structure(df)
costs[["Product_Cost_Pct", "Delivery_Cost_Pct", "Marketing_Cost_Pct",
       "Operating_Cost_Pct", "Margin_Pct"]].round(1)

,Product_Cost_Pct,Delivery_Cost_Pct,Marketing_Cost_Pct,Operating_Cost_Pct,Margin_Pct
Category,,,,,
Electronics,94.10,1.60,3.70,5.50,-4.90
Sports & Fitness,67.00,4.40,3.60,6.40,18.60
Home & Kitchen,65.10,3.90,3.60,6.20,21.20
Fashion,57.20,5.30,3.60,6.70,27.20
Beauty & Personal Care,54.40,7.00,3.70,7.30,27.70


In [20]:
# And the subcategory drill-down: where inside Electronics the loss actually sits.
electronics = df[(df["Category"] == "Electronics") & df["Is_Valid_Revenue"]]
subcats = A.group_economics(electronics, "Subcategory")
subcats["Share_of_Company_Revenue_Pct"] = (
    subcats["Revenue"] / df.loc[df["Is_Valid_Revenue"], "Net_Revenue"].sum() * 100)
subcats[["Orders", "Revenue", "Profit", "Margin_Pct", "Avg_Discount",
         "Share_of_Company_Revenue_Pct"]].round(2)

,Orders,Revenue,Profit,Margin_Pct,Avg_Discount,Share_of_Company_Revenue_Pct
Subcategory,,,,,,
Smartphones,769,"40,731,878.73","-3,696,988.64",-9.08,13.62,31.15
Wearables,765,"13,831,787.96","-22,931.08",-0.17,13.27,10.58
Audio,737,"8,982,641.65","95,861.90",1.07,13.17,6.87
Accessories,765,"2,606,983.65","391,004.61",15.00,13.41,1.99


**Evidence — candidates (a) and (b) are ruled out, (c) is confirmed.**

Electronics' average discount (13.4%) sits within **0.3 percentage points** of
every other category (13.3%–13.5%). Its delivery cost is the *lowest* of the five
as a share of revenue (1.6%, against 7.0% for Beauty & Personal Care), because a
₹90,000 phone and a ₹400 lipstick cost roughly the same to ship. Marketing and
operating costs are near-identical across categories too.

What is not unremarkable is merchandise cost: **94.1% of net revenue in
Electronics**, against 54.4%–67.0% everywhere else. With 94 paise of every rupee
already spent on the goods, there is almost nothing left to cover delivery,
marketing and operations — let alone a discount.

The subcategory table localises it further: **Smartphones are 31.2% of *company*
revenue at −9.1% margin**, losing ₹3.70M — more than the category loses in total,
because Accessories partly offsets it. Company-wide contribution across all three
years is ₹11.6M, so one subcategory is destroying roughly a third of it.

**A limitation worth stating here rather than at the end:** `Product_Cost` is a
single modelled figure. We can prove the ratio is the problem; we cannot tell
whether it comes from supplier terms, from under-pricing against the market, or
from a handful of loss-leader SKUs — and those are three different remedies.

## 7. Investigation: promotions and discount tolerance

If Electronics isn't obviously over-discounted, is the promotional calendar itself
part of the margin story?


In [21]:
promo = (
    valid.groupby("Promotion_Type", observed=True)
    .agg(Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "count"),
         Avg_Discount=("Discount_Percentage", "mean"))
    .assign(Margin_Pct=lambda d: d["Profit"] / d["Revenue"] * 100)
    .reindex(PROMOTION_ORDER)
)
promo


,Revenue,Profit,Orders,Avg_Discount,Margin_Pct
Promotion_Type,,,,,
No Promotion,"45,654,941.54","6,959,096.66",5484,6.56,15.24
Category Offer,"25,661,450.41","2,300,967.31",2855,15.04,8.97
Member Offer,"17,601,475.92","1,522,978.02",1880,14.21,8.65
Festival Sale,"28,731,944.85","1,085,349.78",2959,19.24,3.78
Flash Deal,"13,108,253.83","-263,938.74",1365,24.14,-2.01


In [22]:
fig = px.scatter(promo.reset_index(), x="Avg_Discount", y="Margin_Pct", size="Revenue",
                  text="Promotion_Type", color="Promotion_Type",
                  color_discrete_sequence=CATEGORICAL,
                  title="Deeper average discount tracks straight into negative margin")
fig.update_traces(textposition="top center")
fig.add_hline(y=0, line_color=INK["axis"], line_dash="dot")
fig.update_layout(xaxis_title="Average Discount (%)", yaxis_title="Profit Margin (%)", showlegend=False)
finalize(fig, height=420).show()


The promotion view says *which mechanic* loses money. The more actionable question
is *how much discount each part of the business can absorb before it stops earning* —
because that is a number a pricing system can enforce.

In [23]:
bands = A.discount_band_margin(df, split="Category")
bands["Side"] = bands["Category"].astype(str).where(
    bands["Category"].astype(str) == "Electronics", "Rest of business")
tolerance = (bands.groupby(["Side", "Discount_Band"], observed=True)[["Revenue", "Profit"]].sum()
             .assign(Margin_Pct=lambda d: d["Profit"] / d["Revenue"] * 100))
print(tolerance["Margin_Pct"].unstack(0).round(1).to_string())
print()
print(A.breakeven_discount(df).to_string(index=False))

Side           Electronics  Rest of business
Discount_Band                               
0–5%                  7.90             30.70
5–10%                 3.10             27.70
10–15%               -3.00             24.40
15–20%               -8.50             21.30
20–25%              -14.40             16.10
25%+                -23.30             10.50

              Category Last_Profitable_Band First_Loss_Band
           Electronics                5–10%          10–15%
        Home & Kitchen                 25%+   none observed
               Fashion                 25%+   none observed
      Sports & Fitness                 25%+   none observed
Beauty & Personal Care                 25%+   none observed


In [24]:
fig = px.line(tolerance.reset_index(), x="Discount_Band", y="Margin_Pct", color="Side",
              markers=True, title="Margin by discount depth — the ceiling is category-specific",
              color_discrete_sequence=[CATEGORICAL[0], CATEGORICAL[1]],
              category_orders={"Discount_Band": A.DISCOUNT_BAND_LABELS,
                               "Side": ["Rest of business", "Electronics"]})
fig.add_hline(y=0, line_dash="dot", line_color=INK["axis"], line_width=1,
              annotation_text="break-even", annotation_font_color=INK["muted"])
fig.update_layout(xaxis_title="Discount applied to the order", yaxis_title="Profit margin (%)",
                  legend=dict(orientation="h", y=1.02, x=0, title=None))
finalize(fig, height=420)
fig.show()

**Evidence:** the two halves of the business have completely different discount
tolerance. Electronics earns +7.9% at a 0–5% discount, +3.1% at 5–10%, and is
**loss-making from 10–15% onwards** (−3.0%, then −8.5%, −14.4%, −23.3%). The rest
of the business earns +30.7% at 0–5% and still **+10.5% above a 25% discount**.

₹45.3M of Electronics revenue currently sits in the loss-making bands.

This is the most directly actionable finding in the notebook, because it converts
into a rule rather than a strategy: **the discount ceiling has to be set per
category, from its cost ratio.** One company-wide policy is running two businesses.

*Read this as tolerance, not as a response curve.* Deeply discounted orders differ
from lightly discounted ones in what they contain, not only in their discount. The
policy conclusion survives either reading, which is why it is the one stated.

**Evidence:** orders placed with **no promotion** carry the best margin (15.2%).
**Flash Deal** orders average the deepest discount (24.1%) and are the *only*
promotion type with a **negative margin (−2.0%)** — AuroraCart loses money, on
average, every time it runs a Flash Deal. Festival Sale is marginally positive
(3.8%) but still well below the no-promotion baseline. Category Offer and Member
Offer both sit in a defensible 8–9% range. Promotions are not uniformly bad — Flash
Deals specifically are the one format not paying for itself.


## 8. The "Premium paradox" — and why it is not one

Segments are named for spend behaviour. Does the "Premium" segment behave the way
the name promises?


In [25]:
seg = (
    valid.groupby("Customer_Segment", observed=True)
    .agg(Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "count"),
         Avg_Rating=("Customer_Rating", "mean"))
    .assign(Margin_Pct=lambda d: d["Profit"] / d["Revenue"] * 100,
            AOV=lambda d: d["Revenue"] / d["Orders"])
    .reindex(SEGMENT_ORDER)
)
seg


,Revenue,Profit,Orders,Avg_Rating,Margin_Pct,AOV
Customer_Segment,,,,,,
Premium,"35,346,568.66","1,029,028.32",2728,4.17,2.91,"12,956.95"
Family,"42,507,206.56","4,754,855.00",4971,4.22,11.19,"8,551.04"
Value Seeker,"34,238,377.89","4,085,157.56",4706,4.19,11.93,"7,275.47"
Occasional,"18,665,913.44","1,735,412.15",2138,4.23,9.30,"8,730.55"


In [26]:
fig = px.bar(seg.reset_index(), x="Customer_Segment", y=["AOV"], color_discrete_sequence=[CATEGORICAL[0]],
             title="Premium has the highest average order value...")
fig.update_layout(yaxis_title="Average Order Value (₹)", xaxis_title=None, showlegend=False)
finalize(fig, height=360).show()

fig2 = px.bar(seg.reset_index(), x="Customer_Segment", y="Margin_Pct",
              title="...and the LOWEST profit margin of any segment",
              color="Margin_Pct", color_continuous_scale=SEQUENTIAL_BLUE)
fig2.update_layout(yaxis_title="Profit Margin (%)", xaxis_title=None, coloraxis_showscale=False)
finalize(fig2, height=360).show()


The pooled view above is accurate and would send us after the Premium segment. Before
acting on it, split the same orders by the variable we already know drives margin.

In [27]:
confound = A.segment_margin_confound(df)
confound.round(2)

,Margin_Pooled,Margin_In_Electronics,Margin_Ex_Electronics,Electronics_Share_Pct
Customer_Segment,,,,
Premium,2.91,-6.90,21.87,65.90
Family,11.19,-4.21,22.31,41.94
Value Seeker,11.93,-3.54,24.27,44.36
Occasional,9.30,-3.42,23.49,52.74


In [28]:
split = confound.reset_index().melt(
    id_vars="Customer_Segment", value_vars=["Margin_In_Electronics", "Margin_Ex_Electronics"],
    var_name="Basket", value_name="Margin_Pct")
split["Basket"] = split["Basket"].map({"Margin_In_Electronics": "Electronics orders",
                                       "Margin_Ex_Electronics": "Everything else"})
fig = px.bar(split, x="Customer_Segment", y="Margin_Pct", color="Basket", barmode="group",
             title="The same segments, split by what they actually bought",
             color_discrete_sequence=[CATEGORICAL[1], CATEGORICAL[0]],
             category_orders={"Customer_Segment": SEGMENT_ORDER,
                              "Basket": ["Everything else", "Electronics orders"]})
fig.add_hline(y=0, line_color=INK["axis"], line_width=1)
fig.update_layout(xaxis_title=None, yaxis_title="Profit margin (%)",
                  legend=dict(orientation="h", y=1.02, x=0, title=None))
finalize(fig, height=400)
fig.show()

**Evidence — and a correction to the obvious reading.**

Pooled across all orders, Premium returns a **2.9% margin** against 9.3–11.9% for
every other segment, on the highest average order value in the business (₹12,957).
Spends the most, earns the least: a compelling story, and the wrong one.

Split the same orders by whether they contained Electronics and Premium earns
**21.9% outside it** — within 2.4 points of Family (22.3%), Value Seeker (24.3%)
and Occasional (23.5%) — and loses money inside it, exactly like every other
segment does. The explanation is mix: **65.9% of Premium's revenue is Electronics**,
against 41.9% for Family.

The segment was never the problem. It simply buys the category that is.

**The general rule this produces, which governs the rest of the analysis:** in this
dataset, any comparison that does not control for product category is a comparison
of category mix wearing a disguise. The same trap is waiting in the
acquisition-channel, membership-tier and regional cuts, and each was checked
against it before being reported.

## 9. Investigation: acquisition channel economics

Different channels cost very different amounts to run. Are the expensive ones
earning their keep?


In [29]:
chan = (
    valid.groupby("Acquisition_Channel", observed=True)
    .agg(Revenue=("Net_Revenue", "sum"), Orders=("Order_ID", "count"), Marketing_Cost=("Marketing_Cost", "sum"))
    .assign(Marketing_Cost_Pct_of_Revenue=lambda d: d["Marketing_Cost"] / d["Revenue"] * 100)
    .sort_values("Marketing_Cost_Pct_of_Revenue", ascending=False)
)
chan


,Revenue,Orders,Marketing_Cost,Marketing_Cost_Pct_of_Revenue
Acquisition_Channel,,,,
Marketplace Ads,"18,634,366.42",2021,"1,595,012.74",8.56
Paid Social,"31,210,400.25",3423,"2,144,951.93",6.87
Referral,"14,549,390.92",1616,"359,383.84",2.47
Email,"10,966,798.14",1169,"136,871.16",1.25
Organic Search,"35,837,481.90",4038,"407,005.38",1.14
Direct,"19,559,628.92",2276,"111,457.66",0.57


In [30]:
fig = px.bar(chan.reset_index(), x="Marketing_Cost_Pct_of_Revenue", y="Acquisition_Channel",
             orientation="h", title="Paid channels spend 5-9% of the revenue they bring in on marketing alone",
             color_discrete_sequence=[CATEGORICAL[1]])
fig.update_layout(yaxis=dict(categoryorder="total ascending"), xaxis_title="Marketing Cost as % of Revenue", yaxis_title=None)
finalize(fig, height=380).show()


**Evidence:** Marketplace Ads (8.6%) and Paid Social (6.9%) spend the highest share
of the revenue they generate on marketing cost, while Organic Search and Direct
are nearly free by comparison and *also* bring in the two largest revenue totals.
This doesn't mean paid channels should be cut — they may be reaching customers
organic channels can't — but combined with the margin story above, every paid
acquisition rupee is competing with an already-thinning margin.


## 10. Operations & delivery — what the 43% average hides

Margin is one half of the story. Delivery performance is the other — and it feeds
back into ratings, complaints and returns.


In [31]:
ops = df.groupby("Fulfillment_Mode", observed=True).agg(
    On_Time_Rate=("On_Time_Flag", "mean"), Avg_Rating=("Customer_Rating", "mean"),
    Avg_Delivery_Cost=("Delivery_Cost", "mean"), Orders=("Order_ID", "count"),
).assign(On_Time_Rate=lambda d: d["On_Time_Rate"] * 100).reindex(FULFILLMENT_ORDER)
ops


,On_Time_Rate,Avg_Rating,Avg_Delivery_Cost,Orders
Fulfillment_Mode,,,,
Express Delivery,40.37,4.18,358.84,3287
Standard Delivery,43.73,4.20,270.70,10125
Store Pickup,45.12,4.29,162.78,1558


In [32]:
fig = px.bar(ops.reset_index(), x="Fulfillment_Mode", y="On_Time_Rate",
             title="On-time delivery hovers around 40-45% for every fulfillment mode",
             color_discrete_sequence=[STATUS["warning"]])
fig.add_hline(y=100, line_color=INK["grid"], line_dash="dot")
fig.update_layout(yaxis_title="On-Time Rate (%)", xaxis_title=None, yaxis=dict(range=[0, 100]))
finalize(fig, height=380).show()


In [33]:
print("Correlation, delivery delay (days) vs customer rating:",
      round(df[["Delivery_Delay_Days", "Customer_Rating"]].corr().iloc[0, 1], 3))
print()
complaint_by_ontime = df.groupby("On_Time_Flag")["Complaint_Flag"].mean() * 100
print("Complaint rate when delivered on time:    {:.1f}%".format(complaint_by_ontime[True]))
print("Complaint rate when delivered late:       {:.1f}%".format(complaint_by_ontime[False]))


Correlation, delivery delay (days) vs customer rating: -0.483

Complaint rate when delivered on time:    4.0%
Complaint rate when delivered late:       12.3%


The mode-level chart above pools three years. Every mode lands near 43%, which
reads as a function that is uniformly broken — and makes delivery the natural place
to cut cost.

The case tells us the logistics network was **restructured in January 2025**. That
is the second dated intervention, and the same before/after test applies.

In [34]:
logistics = A.logistics_comparison(df)
logistics[["Orders", "On_Time_Pct", "Delivery_Delay_Days", "Delivery_Cost_Per_Order",
           "Avg_Rating", "Complaint_Pct", "Return_Pct"]].round(2).T

Era,Before new contract,After new contract
Orders,"8,617.00","6,353.00"
On_Time_Pct,34.40,55.00
Delivery_Delay_Days,0.74,0.43
Delivery_Cost_Per_Order,259.87,304.51
Avg_Rating,4.15,4.27
Complaint_Pct,10.11,6.86
Return_Pct,7.22,7.13


In [35]:
monthly_ops = A.monthly_operations(df).reset_index()
pooled = df["On_Time_Flag"].mean() * 100

fig = px.line(monthly_ops, x="Order_YearMonth", y="On_Time_Pct", markers=True,
              title="On-time rate by month — the average is two stories averaged together",
              color_discrete_sequence=[CATEGORICAL[0]])
for _, row in monthly_ops[monthly_ops["Is_Peak"]].iterrows():
    fig.add_vrect(x0=row["Order_YearMonth"].isoformat(),
                  x1=(row["Order_YearMonth"] + pd.offsets.MonthEnd(1)).isoformat(),
                  fillcolor=STATUS["critical"], opacity=0.07, line_width=0, layer="below")
fig.add_vline(x=A.LOGISTICS_START.isoformat(), line_dash="dash", line_color=INK["muted"],
              line_width=1, annotation_text="new logistics contract",
              annotation_font_color=INK["secondary"])
fig.add_hline(y=pooled, line_dash="dot", line_color=INK["axis"], line_width=1,
              annotation_text=f"pooled average: {pooled:.0f}%", annotation_position="bottom right",
              annotation_font_color=INK["muted"])
fig.update_layout(xaxis_title=None, yaxis_title="On-time rate (%)", yaxis_range=[0, 100])
finalize(fig, height=420)
fig.show()

In [36]:
# The shaded bands: October-November, every year.
A.peak_season_gap(df).round(1)

Window,Oct–Nov peak,Rest of year,Gap_pp
Year,,,
2023,16.80,38.20,21.40
2024,18.60,39.80,21.20
2025,34.00,60.70,26.70


**Evidence — and a second correction.**

The pooled 43% is the average of a success and a recurring failure, and describes
neither.

**The logistics contract worked.** On-time delivery went **34.4% → 55.0%** after
January 2025, average delay 0.74 → 0.43 days, complaints **10.1% → 6.9%**, average
rating 4.15 → 4.27. It cost about **₹45 more per order** — 0.26 points of revenue.
Every fulfilment mode improved, including Store Pickup. This is the one function in
the business with a measurable, funded improvement, and the summary table hides it.

**Peak season still breaks it.** On-time collapses to roughly 34% every
October–November: a 26.7-point gap against the rest of 2025, and the same pattern
in 2023 (21.4 points) and 2024 (21.2). That is a capacity problem, it is entirely
predictable, and the contract did not fix it.

**The experience link.** Delivery delay correlates negatively with rating
(r ≈ −0.48), and a late delivery **triples** the complaint rate (12.3% vs 4.0%).

**A note on causation:** the delay↔rating relationship is an *association*, not
proof that delay alone causes low ratings — customers already dissatisfied for
other reasons may also be likelier to notice and penalise a delay. Its direction
and consistency across fulfilment modes make delay a credible driver, but not one
this dataset can isolate from confounds.

## 11. Ranking the drivers — which lens actually splits profitability?

The leadership meeting proposed four competing explanations: geography, customer
type, acquisition channel and fulfilment. Rather than argue them one at a time,
measure all of them on the same scale.

For each candidate dimension, the **revenue-weighted standard deviation of group
margin** around the company margin, `sqrt(Σ wᵍ (marginᵍ − m)²)`. Weighting by
revenue share is the point: an unweighted range lets a thin, wild group outrank a
dimension that splits half the business. `Profit_Gap_Mn` converts that spread into
rupees — what below-average groups would add if each merely reached the company
margin.

In [37]:
drivers = A.driver_ranking(df)
drivers.round(2)

,Dimension,Levels,Worst_Margin_pp,Best_Margin_pp,Margin_Range_pp,Weighted_SD_pp,Profit_Gap_Mn
0,Subcategory,19,-9.08,31.11,40.19,15.10,9.26
1,Category,5,-4.89,27.67,32.56,14.15,9.10
2,Price_Band,4,-6.89,24.14,31.03,13.24,8.06
3,Promotion_Type,5,-2.01,15.24,17.26,5.64,2.93
4,Coupon_Used,2,4.89,13.28,8.39,4.19,2.74
5,Acquisition_Channel,6,2.59,13.53,10.94,3.83,2.32
6,Customer_Segment,4,2.91,11.93,9.02,3.72,2.11
7,Brand_Tier,3,3.99,12.86,8.88,3.33,1.87
8,New_vs_Returning,2,6.97,10.42,3.44,1.71,1.11
9,Membership_Type,4,6.81,9.81,3.00,1.00,0.58


In [38]:
ranked = drivers.sort_values("Weighted_SD_pp")
fig = px.bar(ranked, x="Weighted_SD_pp", y=ranked["Dimension"].str.replace("_", " "),
             orientation="h", title="What actually splits profitability",
             color_discrete_sequence=[CATEGORICAL[0]],
             text=ranked["Weighted_SD_pp"].round(1).astype(str) + " pts")
fig.update_traces(textposition="outside", textfont_color=INK["secondary"])
fig.update_layout(xaxis_title="Revenue-weighted spread in margin (percentage points)",
                  yaxis_title=None, margin=dict(l=170),
                  xaxis_range=[0, ranked["Weighted_SD_pp"].max() * 1.22])
fig.update_yaxes(showgrid=False)
finalize(fig, height=520)
fig.show()

**Evidence:** product mix dominates by roughly a factor of twenty.

| Dimension | Weighted spread | Profit gap |
|---|---|---|
| Subcategory | 15.1 pts | ₹9.3M |
| Category | 14.2 pts | ₹9.1M |
| Price band | 13.2 pts | ₹8.1M |
| Promotion type | 5.6 pts | ₹2.9M |
| Acquisition channel | 3.8 pts | ₹2.3M |
| Customer segment | 3.7 pts | ₹2.1M |
| **Region** | **0.8 pts** | ₹0.4M |
| **Fulfilment mode** | **0.6 pts** | ₹0.3M |

Category, subcategory and price band are one finding, not three: 74.9% of
Electronics revenue is in the Premium price band, so the Premium band's −6.9%
margin is Electronics restated.

The three dimensions the leadership meeting spent most of its time on — geography,
fulfilment mode and customer type — between them explain almost none of the
variation in profitability. That is arguably the most useful single output of this
analysis, because it tells the room what *not* to spend the next quarter on.

## 12. Synthesis — the story end to end

**Context.** Net revenue nearly doubled, ₹29.3M → ₹58.0M (+98%), across 14,970
orders and 6,778 customers. The kind of trajectory a board deck leads with.

**Tension.** Margin fell 12.4% → 8.9% → 7.1%. Measured against the growth programme
that produced it, revenue per month rose 77% while contribution per month rose 14%.
Growth was purchased, not earned.

**Investigation & evidence.**

1. **It is erosion, not mix.** 92% of the 5.27-point decline is within-category;
   holding the 2023 mix fixed, 2025 margin still lands at 7.6%. (§5)
2. **Electronics is the whole story.** 50.6% of net revenue at −4.9% margin, and
   Smartphones alone are 31.2% of company revenue at −9.1%, losing ₹3.70M against
   ₹11.6M of company-wide contribution. (§6)
3. **The mechanism is cost, not discounting.** Merchandise cost is 94.1% of net
   revenue in Electronics against 54–67% elsewhere; discount depth is within 0.3
   points across all five categories. (§6)
4. **Discount tolerance is category-specific.** Electronics turns loss-making above
   a 10% discount; the rest of the business earns 10.5% even above 25%. Flash Deals
   (24.1% average discount, 10.0% of revenue) are the only promotion type with a
   negative margin. (§7)
5. **The "Premium segment problem" is a mix artifact.** 2.9% pooled, 21.9% outside
   Electronics — 65.9% of its revenue is Electronics. Any cut that does not control
   for category is a category-mix comparison in disguise. (§8)
6. **Paid acquisition compounds it.** Marketplace Ads and Paid Social spend
   6.9–8.6% of the revenue they generate on marketing and lost the most margin
   after Accelerate 2.0, on 38% of revenue between them. (§9)
7. **Delivery is the one thing working.** The January 2025 contract took on-time
   34.4% → 55.0% and complaints 10.1% → 6.9% for ~₹45 per order. The 43%
   company-wide figure hides both that and a festive collapse to ~34% every
   October–November. (§10)
8. **Product mix outranks every other lens by ~20×.** Region and fulfilment mode
   barely separate profitability at all. (§11)

**Consequence.** If nothing changes, the growth narrative keeps outrunning value
creation, and the largest single pool of destroyed contribution — half the revenue
base sold at a loss — keeps scaling with it.

## 13. Recommendations

*Three, ranked by contribution at stake, then controllability, then strength of
evidence. The full write-up — expected benefit, principal risk, and the one thing
we would ask for before committing — is in `docs/recommendations.md`, and the same
three appear on the dashboard's **Decision** tab.*

1. **Re-cost and re-price Electronics before scaling it further.** Half of revenue
   at −4.9% margin, driven by a 94.1% merchandise-cost ratio. Moving it to
   break-even recovers ~₹3.2M of contribution — a ~28% increase on the three-year
   total — without touching the four healthy categories. *Risk:* Electronics may be
   the traffic and attachment engine. *We would ask for:* SKU-level cost of goods
   and a competitor price index.

2. **Set a category-aware discount ceiling; restructure or retire Flash Deals.**
   The only recommendation here that is a *rule* rather than a bet — enforceable in
   the pricing system next quarter. ₹45.3M of Electronics revenue currently sits in
   loss-making discount bands. *Risk:* promotion is partly defensive; withdrawn
   volume may not convert to full price. *We would ask for:* promotion
   incrementality from a holdout.

3. **Underwrite acquisition on contribution, not revenue.** Every channel lost
   margin after Accelerate 2.0; the paid ones lost most and cost most. *Risk:* this
   is the weakest-evidenced of the three — `Marketing_Cost` is allocated, and
   order-level data has no lifetime view. It ranks third for exactly that reason.
   *We would ask for:* cohort retention and repeat value by channel.

**What we are explicitly not recommending:** cutting delivery investment (it is the
one function that measurably improved), a customer-segment programme (§8 shows the
segment signal is a mix artifact), or a regional strategy (§11 — region separates
margin by 0.8 points). The October–November on-time collapse is a real operating
item with a named owner, but its contribution impact is an order of magnitude below
the three above.

## 14. Limitations

- **Synthetic dataset.** Patterns are internally consistent (cancellation ↔ zero
  revenue holds 100% of the time), but this is not real transactional data. Treat
  the conclusions as illustrative of the *method*.
- **Returns are not netted out of revenue.** `Return_Flag` does not reduce
  `Net_Revenue` or `Profit`, so every margin here is an upper bound — and unevenly
  so, since return rates run 4.9% (Beauty & Personal Care) to 10.7% (Fashion).
  Netting them out would penalise a *healthy* category hardest and narrow the gap
  to Electronics somewhat, without reversing it.
- **No SKU-level cost breakdown.** We can show *that* Electronics is unprofitable
  and *that* the ratio is the mechanism, but not decompose it into sourcing terms
  vs. pricing vs. specific loss-leader SKUs.
- **Marketing and operating costs are allocated, not observed** at the order. This
  barely affects §6 (a direct merchandise cost) and materially affects §9, which is
  why the acquisition recommendation ranks third.
- **Correlation, not causation** on delivery delay → rating (§10): a believable
  driver, not an isolated, proven one.
- **Before/after, not difference-in-differences.** Neither Accelerate 2.0 nor the
  logistics contract has a control group; other things changed in the same months.
- **Structural missingness is labelled, not imputed** (§1). Nothing here fills a
  gap with an invented value, so counts differ between charts that use different
  columns — deliberately.